In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.listdir('/content/drive/MyDrive/Colab')

['placement_dataset.csv',
 'placement_predict_50k Dataset (1) (1).csv',
 'placement_predict_50k_adjusted.csv',
 'age_insurance.csv',
 'simple_placement_dataset.csv',
 'movies_dataset(1).csv',
 'WineQT.csv',
 'WineQT_Feature_Engineered.csv',
 'placement_predict_50k_adjusted (1).csv',
 'student_data.csv',
 'xgboost_lightgbm_shap_student_dataset.csv',
 'iris_dataset.csv',
 'breast_cancer_classification_dataset (1).csv',
 'mall_customers_clustering_dataset.csv',
 'digits_pca_tsne_umap_dataset.csv',
 'isolation_forest_dataset.csv',
 'socioeconomic_profiling_dataset.csv',
 'ml_cross_validation_calibration_dataset.csv',
 'hyperparameter_tuning_dataset.csv',
 'evaluation_threshold_calibration_dataset.csv',
 'student_performance.csv',
 'student_performance_mlops.csv',
 'dimensionality_reduction_anomaly_detection_dataset.csv',
 'data_leakage_cross_validation_nested_cv_dataset.csv']

In [3]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab/placement_dataset.csv')

In [5]:
# Data Leakage, Cross-Validation, and Nested CV
# Machine Learning Classification Project

# Install if required:
# pip install pandas numpy scikit-learn matplotlib seaborn

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# =========================================================
# 1. Load dataset
# =========================================================
df = pd.read_csv('/content/drive/MyDrive/Colab/data_leakage_cross_validation_nested_cv_dataset.csv')

print("First 5 rows:")
print(df.head())

print("\nDataset shape:", df.shape)
print("\nClass distribution:")
print(df["Passed"].value_counts())

# Student_ID is only an identifier
X = df.drop(columns=["Student_ID", "Passed"])
y = df["Passed"]

# =========================================================
# 2. Train-Test Split
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# =========================================================
# 3. DATA LEAKAGE DEMONSTRATION
# =========================================================
# WRONG APPROACH:
# Scaling the complete dataset BEFORE cross-validation.
# The scaler sees information from validation folds.
#
# This can cause data leakage because information from
# validation data is used during preprocessing.

scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)

print("\nWARNING: Scaling before cross-validation is a leakage-prone approach.")

# =========================================================
# 4. CORRECT APPROACH: Pipeline
# =========================================================
# The scaler is fitted separately inside every CV training fold.
# Therefore validation data is not used to fit the scaler.

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# =========================================================
# 5. Standard 5-Fold Cross-Validation
# =========================================================
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=outer_cv,
    scoring="accuracy"
)

print("\n5-Fold Cross-Validation Scores:")
print(cv_scores)

print("Mean CV Accuracy:", cv_scores.mean())
print("CV Standard Deviation:", cv_scores.std())

# =========================================================
# 6. Train on training data and evaluate test data
# =========================================================
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("\nTest Set Performance")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# =========================================================
# 7. NESTED CROSS-VALIDATION
# =========================================================
# Inner CV:
# Selects the best hyperparameters.
#
# Outer CV:
# Gives an unbiased estimate of model performance.

inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=10
)

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

nested_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

grid_search = GridSearchCV(
    estimator=nested_pipeline,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="accuracy"
)

nested_scores = cross_val_score(
    grid_search,
    X_train,
    y_train,
    cv=outer_cv,
    scoring="accuracy"
)

print("\nNested Cross-Validation Scores:")
print(nested_scores)

print("Mean Nested CV Accuracy:", nested_scores.mean())
print("Nested CV Standard Deviation:", nested_scores.std())

# =========================================================
# 8. Fit final model using inner CV
# =========================================================
grid_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid_search.best_params_)

print("Best Inner-CV Score:")
print(grid_search.best_score_)

# =========================================================
# 9. Final Test Evaluation
# =========================================================
final_pred = grid_search.predict(X_test)

print("\nFinal Test Accuracy:",
      accuracy_score(y_test, final_pred))

print("\nFinal Classification Report:")
print(classification_report(y_test, final_pred))

# =========================================================
# 10. Save predictions
# =========================================================
results = X_test.copy()
results["Actual_Passed"] = y_test.values
results["Predicted_Passed"] = final_pred

results.to_csv(
    "nested_cv_predictions.csv",
    index=False
)

print("\nCompleted successfully.")
print("Predictions saved as nested_cv_predictions.csv")


First 5 rows:
  Student_ID  Study_Hours  Attendance_Percentage  Assignment_Score  \
0       S001         6.49                  87.08             73.75   
1       S002         5.22                  66.93             65.99   
2       S003         6.80                  88.44             73.71   
3       S004         8.55                  94.27             79.61   
4       S005         5.03                  82.96             72.68   

   Internal_Score  Sleep_Hours  Previous_Score  Screen_Time_Hours  Passed  
0           98.52         5.93           37.48               3.35       1  
1           69.09         7.01           54.23               3.56       1  
2           59.37         6.14           61.38               9.10       1  
3           47.29         6.47           98.20               5.00       1  
4           42.95         8.81           76.90               6.56       1  

Dataset shape: (600, 9)

Class distribution:
Passed
1    596
0      4
Name: count, dtype: int64

Training sa

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predic


Nested Cross-Validation Scores:
[1.         1.         0.98958333 0.98958333 0.98958333]
Mean Nested CV Accuracy: 0.99375
Nested CV Standard Deviation: 0.00510310363079827


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(



Best Parameters:
{'model__C': 0.01, 'model__solver': 'liblinear'}
Best Inner-CV Score:
0.99375

Final Test Accuracy: 0.9916666666666667

Final Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.99      1.00      1.00       119

    accuracy                           0.99       120
   macro avg       0.50      0.50      0.50       120
weighted avg       0.98      0.99      0.99       120


Completed successfully.
Predictions saved as nested_cv_predictions.csv


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
